<a href="https://colab.research.google.com/github/collinskiplangatrotich/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/excel_duka_branch_summary_xlsx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill

# 1. Generate Duka Electronics data (seed 2026) and calculate revenue
rng = np.random.default_rng(2026)
n = 1000
products = {"Phone": 15000, "Laptop": 65000, "Headphones": 2500, "Tablet": 28000, "Charger": 1200}

df = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"], n, p=[0.4, 0.2, 0.15, 0.15, 0.1]),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(["In-store", "Online", "Phone order"], n, p=[0.55, 0.35, 0.10]),
})
df["unit_price"] = df["product"].map(products)
df["revenue"] = df["quantity"] * df["unit_price"]

# 2. Create Excel workbook
os.makedirs("excel", exist_ok=True)
wb = openpyxl.Workbook()

# Sheet 1: Branch_Summary (Formulas & Lookup)
ws_sum = wb.active
ws_sum.title = "Branch_Summary"

# Region Lookup Table (Columns F:G)
ws_sum["F1"] = "Branch"
ws_sum["G1"] = "Region"
regions = [
    ("Nairobi", "Nairobi Metro"),
    ("Mombasa", "Coast"),
    ("Nakuru", "Rift Valley"),
    ("Kisumu", "Nyanza / Western"),
    ("Eldoret", "North Rift"),
]
for idx, (b, r) in enumerate(regions, start=2):
    ws_sum[f"F{idx}"] = b
    ws_sum[f"G{idx}"] = r

# Main Branch Revenue Table (Columns A:D)
headers = ["Branch", "Region (XLOOKUP)", "Total Revenue (SUMIFS)", "Order Count (COUNTIF)"]
for col_idx, h in enumerate(headers, start=1):
    cell = ws_sum.cell(row=1, column=col_idx, value=h)
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="1F4E78")

branches_ordered = ["Nairobi", "Mombasa", "Nakuru", "Kisumu", "Eldoret"]
for row_idx, branch_name in enumerate(branches_ordered, start=2):
    ws_sum[f"A{row_idx}"] = branch_name
    ws_sum[f"B{row_idx}"] = f"=_xlfn.XLOOKUP(A{row_idx}, $F$2:$F$6, $G$2:$G$6)"
    ws_sum[f"C{row_idx}"] = f"=SUMIFS(Sales_Data!$H$2:$H$1001, Sales_Data!$C$2:$C$1001, A{row_idx})"
    ws_sum[f"D{row_idx}"] = f"=COUNTIF(Sales_Data!$C$2:$C$1001, A{row_idx})"
    ws_sum[f"C{row_idx}"].number_format = "#,##0"

# Grand Total Row
ws_sum["A7"] = "Grand Total"
ws_sum["A7"].font = Font(bold=True)
ws_sum["C7"] = "=SUM(C2:C6)"
ws_sum["C7"].font = Font(bold=True)
ws_sum["C7"].number_format = "#,##0"
ws_sum["D7"] = "=SUM(D2:D6)"
ws_sum["D7"].font = Font(bold=True)

# Sheet 2: Sales_Data (Source records for SUMIFS & PivotTable)
ws_data = wb.create_sheet(title="Sales_Data")
export_cols = ["order_id", "order_date", "branch", "product", "quantity", "channel", "unit_price", "revenue"]
ws_data.append(export_cols)

df_export = df[export_cols].copy()
df_export["order_date"] = df_export["order_date"].dt.strftime("%Y-%m-%d")
for row in df_export.itertuples(index=False):
    ws_data.append(list(row))

wb.save("excel/duka_branch_summary.xlsx")
print("Saved excel/duka_branch_summary.xlsx successfully!")

Saved excel/duka_branch_summary.xlsx successfully!
